# Econ 3916 PS4 Warm-up — the code PS4 asks you to write

**Not graded.** Practice for Problem Set 4, on different data. Each exercise
shows you working code (*Watch*), asks you to write a similar piece yourself
(*Your turn*), then checks it. The exercises follow PS4's order: R-squared in and out of sample, what extra columns do to R-squared, residuals, who gets over-forecast, a near-duplicate column, the complexity sweep, choosing a model by cross-validation, then fitting the preprocessing inside each fold.

- **Turn AI code completion off** (Colab: *Settings › AI assistance*), as PS4
  Phases 1–3 require. You may still click **Explain error**.
- The *Watch* cells use farms (and their harvest) and the *Your turn* cells use hospitals (and their patient visits). The two
  differ **on purpose**: you write each piece yourself instead of copying it.
- Run the cells in order: later exercises use what earlier ones made.
- If a check fails, read its message, fix your cell, and run both again.
- Stuck for more than five minutes? Look back at the *Watch* cell above it.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.pipeline import Pipeline

---
## 1. R-squared in and out of sample (PS4 Step 1.2)

R-squared on the rows a model was fitted to says how well it fits **those** rows. With many columns and not many rows, OLS also fits part of the noise, and that part does not come back on new rows. So split first, fit on the training rows only, and score on the rows the model never saw. `sm.add_constant(X)` adds the intercept column; `fit.predict(X_test)` gives predictions; `r2_score(actual, predicted)` scores them.

**Watch:** run this and read it.

In [ ]:
rng = np.random.default_rng(7)
n_farms = 120
farms = pd.DataFrame({"area": rng.uniform(5, 100, n_farms), "rain": rng.normal(600, 100, n_farms)})
for j in range(25):
    farms[f"junk_{j}"] = rng.normal(size=n_farms)          # 25 columns that have nothing to do with the harvest
harvest = 3 * farms["area"] + 0.1 * farms["rain"] + rng.normal(0, 25, n_farms)

X_tr, X_te, y_tr, y_te = train_test_split(sm.add_constant(farms), harvest, test_size=0.3, random_state=0)
farm_fit = sm.OLS(y_tr, X_tr).fit()                        # fitted on the 84 training farms only
print(f"R-squared on the {len(y_tr)} training farms: {farm_fit.rsquared:.3f}")
print(f"R-squared on the {len(y_te)} held-out farms: {r2_score(y_te, farm_fit.predict(X_te)):.3f}")

**Your turn:** for 150 hospitals, build a DataFrame `hosp` with `beds` (uniform 20 to 400), `staff_ratio` (Normal(1.5, 0.3)) and 30 columns of pure noise, `junk_0` to `junk_29`. Make `visits = 40 * beds + 2000 * staff_ratio + Normal(0, 2000)`. Split with `test_size=0.3, random_state=0`, fit OLS with every column on the training rows, and store the training R-squared as `r2_in` and the held-out R-squared as `r2_out`.

In [ ]:
rng = np.random.default_rng(11)
n_hosp = 150
# YOUR CODE HERE
hosp = visits = None
r2_in = r2_out = None

In [ ]:
assert hosp is not None and hosp.shape == (150, 32), "hosp needs 150 rows and 32 columns: beds, staff_ratio and 30 junk columns"
assert r2_in is not None and r2_in > 0.8, "the training R-squared should be high: fit on the training rows with every column"
assert r2_out is not None and r2_out < r2_in - 0.03, \
    "the held-out R-squared should come out lower: score the model on the rows it never saw"
print(f"training R-squared {r2_in:.3f} | held-out R-squared {r2_out:.3f} | gap {r2_in - r2_out:.3f}")
print("The first number is the one a handover note quotes. The second is the one next month will look like.")

---
## 2. Extra columns never lower R-squared (PS4 Step 1.3)

OLS picks the coefficients that make the squared residuals as small as possible. A new column could always get a coefficient of 0, so the fit with it is at least as good: **R-squared cannot fall** when you add a column, even a column of random numbers. That is why it cannot choose between models. Adjusted R-squared (`fit.rsquared_adj`) charges for each column, so pure noise lowers it more often than not, but not every time.

**Watch:** run this and read it.

In [ ]:
base = sm.OLS(harvest, sm.add_constant(farms[["area", "rain"]])).fit()
noisy = farms[["area", "rain"]].copy()
for j in range(3):
    noisy[f"noise_{j}"] = rng.normal(size=n_farms)        # three columns of pure noise
more = sm.OLS(harvest, sm.add_constant(noisy)).fit()
print(f"{'':>16}{'R-squared':>11}{'adjusted':>11}")
print(f"{'area + rain':>16}{base.rsquared:>11.4f}{base.rsquared_adj:>11.4f}")
print(f"{'+ 3 noise':>16}{more.rsquared:>11.4f}{more.rsquared_adj:>11.4f}")

**Your turn:** fit `visits` on `beds` and `staff_ratio` only (all 150 hospitals), as `base`. Then 200 times: add three columns of pure noise (`rng.normal(size=(150, 3))`), refit, and record whether R-squared went up (or stayed equal) and whether adjusted R-squared went down. Store the two shares as `share_r2_up` and `share_adj_down`.

In [ ]:
rng = np.random.default_rng(12)
# YOUR CODE HERE
share_r2_up = share_adj_down = None

In [ ]:
assert share_r2_up is not None and share_r2_up == 1.0, \
    "R-squared went down in some draw: refit on beds, staff_ratio AND the noise, and compare with >="
assert share_adj_down is not None and 0.4 < share_adj_down < 0.85, \
    "adjusted R-squared should fall in most draws but not all: compare each refit's rsquared_adj with base's"
print(f"R-squared rose in {share_r2_up:.0%} of 200 draws; adjusted R-squared fell in {share_adj_down:.0%}")
print("A model with more columns always looks better in-sample. Choose with held-out error instead.")

---
## 3. Residuals whose spread grows (PS4 Step 2.1)

Plot residuals against fitted values. A fan, narrow on one side and wide on the other, is non-constant variance. The coefficient is still right on average; the **standard error** is what goes wrong, because the classical formula assumes one spread for every row. `fit(cov_type="HC3")` keeps the same estimate and computes a standard error that allows the spread to change (Lab 14).

**Watch:** run this and read it.

In [ ]:
rng = np.random.default_rng(8)
area = rng.lognormal(3.5, 0.7, 500)
crop = 20 + 3 * area + rng.normal(0, 0.8 * area)          # the error's spread grows with area
Xa = sm.add_constant(area)
classical = sm.OLS(crop, Xa).fit()
robust = sm.OLS(crop, Xa).fit(cov_type="HC3")

plt.scatter(classical.fittedvalues, classical.resid, s=8, alpha=0.5)
plt.axhline(0, color="firebrick")
plt.xlabel("fitted harvest"); plt.ylabel("residual"); plt.title("The spread widens with farm size")
plt.show()
print(f"slope {classical.params[1]:.3f} in both | classical SE {classical.bse[1]:.3f} | HC3 SE {robust.bse[1]:.3f}")

**Your turn:** for 500 hospitals, draw `beds` as lognormal(5, 0.7) and `visits3 = 5000 + 40 * beds + Normal(0, 10 * beds)`. Fit OLS of `visits3` on `beds` twice, classical as `fit_c` and HC3 as `fit_r`, and plot the residuals against the fitted values. Store the two standard errors of the `beds` slope as `se_classical` and `se_robust`, and the residual standard deviation in the top quarter of `beds` divided by the one in the bottom quarter (use `pd.qcut(beds, 4, labels=False)`) as `spread_ratio`.

In [ ]:
rng = np.random.default_rng(13)
# YOUR CODE HERE
fit_c = fit_r = None
se_classical = se_robust = spread_ratio = None

In [ ]:
assert fit_c is not None and fit_r is not None, "fit the model twice: classical (fit_c) and HC3 (fit_r)"
assert np.isclose(np.asarray(fit_c.params)[1], np.asarray(fit_r.params)[1]), \
    "the two fits should give the SAME slope: HC3 changes the standard error, not the estimate"
assert se_robust > 1.3 * se_classical, "with this much fanning the HC3 standard error should be clearly larger"
assert spread_ratio > 2, "the residuals of the biggest hospitals should be far more spread out than the smallest"
print(f"slope {np.asarray(fit_c.params)[1]:.2f} both ways | classical SE {se_classical:.3f} | HC3 SE {se_robust:.3f}")
print(f"residual spread, top quarter vs bottom quarter of beds: {spread_ratio:.1f} times")

---
## 4. Who gets over-forecast (PS4 Step 2.2)

A straight line through a relationship that flattens misses in a pattern: too high at the ends, too low in the middle. To see it, group the residuals (actual minus predicted) by size with `pd.qcut` and take the mean in each group. A **negative** mean is an over-forecast. Divide it by the group's average actual value to state it in percent.

**Watch:** run this and read it.

In [ ]:
rng = np.random.default_rng(9)
fields = pd.DataFrame({"area": rng.uniform(5, 100, 400), "rain": rng.normal(600, 100, 400)})
yield_t = 50 + 6 * fields["area"] - 0.03 * fields["area"] ** 2 + 0.1 * fields["rain"] + rng.normal(0, 20, 400)
line = sm.OLS(yield_t, sm.add_constant(fields)).fit()      # a straight line in area

miss = yield_t - line.fittedvalues                          # actual minus predicted
size_group = pd.qcut(fields["area"], 5, labels=["smallest", "2", "3", "4", "largest"])
print(miss.groupby(size_group, observed=True).mean().round(1))

**Your turn:** for 400 hospitals, draw `beds` uniform(50, 400) and `staff` Normal(300, 60), and make `visits4 = 2000 + 60 * beds - 0.08 * beds**2 + 20 * staff + Normal(0, 800)`. Fit a straight line in `beds` and `staff`. Store the mean residual in each fifth of `beds` (smallest first) as the Series `by_size`, and the largest fifth's mean residual as a percent of its mean `visits4` as `top_pct`.

In [ ]:
rng = np.random.default_rng(14)
# YOUR CODE HERE
by_size = top_pct = None

In [ ]:
assert by_size is not None and len(by_size) == 5, "by_size needs one mean residual per fifth of beds"
assert by_size.iloc[-1] < 0, \
    "the largest hospitals should be OVER-forecast (negative mean residual): residual = actual minus predicted"
assert by_size.iloc[2] > 0, "the middle fifth should be under-forecast: a straight line through a curve that flattens"
assert top_pct is not None and -10 < top_pct < 0, "top_pct: the largest fifth's mean residual over its mean visits, times 100"
print(by_size.round(0).to_string())
print(f"the largest fifth is over-forecast by {-by_size.iloc[-1]:,.0f} visits, {-top_pct:.1f}% of what they get")

---
## 5. A near-duplicate column and its VIF (PS4 Step 2.2)

A variance inflation factor (VIF) says how much of a column the other columns already explain: above about 10, the column is nearly a combination of the others. What it damages: that column's **own coefficient**, whose standard error balloons because the data cannot tell it apart from the others. What it does not damage: the fit and the forecasts. Dropping a high-VIF column because of its VIF does not make the model predict better. `variance_inflation_factor(X, i)` takes the matrix with the constant and the position of the column.

**Watch:** run this and read it.

In [ ]:
rng = np.random.default_rng(10)
inputs = pd.DataFrame({"fertilizer": rng.normal(100, 20, 300), "seed_cost": rng.normal(50, 10, 300)})
inputs["total_inputs"] = inputs["fertilizer"] + inputs["seed_cost"] + rng.normal(0, 2, 300)   # nearly their sum
crop5 = 2 * inputs["fertilizer"] + 3 * inputs["seed_cost"] + rng.normal(0, 30, 300)

Xi = sm.add_constant(inputs)
print({c: round(float(variance_inflation_factor(Xi.to_numpy(), i)), 1) for i, c in enumerate(Xi.columns) if c != "const"})
full = sm.OLS(crop5, Xi).fit()
lean = sm.OLS(crop5, sm.add_constant(inputs[["fertilizer", "seed_cost"]])).fit()
print(f"R-squared {full.rsquared:.4f} with total_inputs, {lean.rsquared:.4f} without")
print(f"SE of fertilizer {full.bse['fertilizer']:.3f} with total_inputs, {lean.bse['fertilizer']:.3f} without")

**Your turn:** for 400 hospitals, draw `beds` uniform(50, 400), `nurses` Normal(200, 40), `doctors` Normal(60, 15), and `staff_total = nurses + doctors + Normal(0, 3)`, all in a DataFrame. Make `visits5 = 30 * beds + 15 * nurses + 40 * doctors + Normal(0, 1500)`. Store the four VIFs as a Series `vifs` indexed by column name. Fit with all four columns (`fit_all`) and without `staff_total` (`fit_drop`); store `r2_change = fit_all.rsquared - fit_drop.rsquared` and `se_ratio`, the standard error of `nurses` with `staff_total` divided by the one without.

In [ ]:
rng = np.random.default_rng(15)
# YOUR CODE HERE
vifs = fit_all = fit_drop = None
r2_change = se_ratio = None

In [ ]:
assert vifs is not None and len(vifs) == 4, "vifs needs one VIF per predictor (not the constant)"
assert vifs["staff_total"] > 10 and vifs["beds"] < 2, "staff_total is nearly nurses + doctors, so its VIF is high; beds is not"
assert r2_change is not None and 0 <= r2_change < 0.01, "dropping the near-duplicate should barely change the fit"
assert se_ratio is not None and se_ratio > 3, "with staff_total in, the SE of nurses should be several times larger"
print(vifs.round(1).to_string())
print(f"R-squared changes by {r2_change:.4f}; the SE of nurses is {se_ratio:.1f} times larger with staff_total in")
print("High VIF blurs one coefficient. It does not make the forecasts worse.")

---
## 6. The complexity sweep (PS4 Step 3.1)

Fit models of rising flexibility and score each twice: on the rows it was fitted to and on new rows. Training error can only fall as the model gets more flexible. Held-out error falls, then turns: on the left of the turn the model is too simple to follow the curve (**bias**), on the right it starts to follow the noise in its training rows (**variance**). `np.polyfit(x, y, d)` fits a degree-d polynomial and `np.polyval(c, x)` predicts from it, as in Lab 15.

**Watch:** run this and read it.

In [ ]:
rng = np.random.default_rng(16)
def farm_truth(x):
    return np.cos(x) + 0.2 * x                              # the curve a real analyst never sees
xf = rng.uniform(0, 6, 70);  yf = farm_truth(xf) + rng.normal(0, 0.4, 70)        # 70 training farms
xf_new = rng.uniform(0, 6, 2_000);  yf_new = farm_truth(xf_new) + rng.normal(0, 0.4, 2_000)

tr, te = [], []
for d in range(1, 13):
    c = np.polyfit(xf, yf, d)
    tr.append(np.mean((np.polyval(c, xf) - yf) ** 2))
    te.append(np.mean((np.polyval(c, xf_new) - yf_new) ** 2))
plt.plot(range(1, 13), tr, marker="o", label="training error")
plt.plot(range(1, 13), te, marker="o", label="held-out error")
plt.xlabel("polynomial degree"); plt.ylabel("mean squared error"); plt.yscale("log"); plt.legend(); plt.show()
print(f"held-out error is lowest at degree {int(np.argmin(te)) + 1}")

**Your turn:** the distance a patient travels, `x`, is uniform(0, 6) and a visit index is `y = np.sin(x) + 0.3 * x + Normal(0, 0.4)`. Draw 80 training hospitals (`x_tr`, `y_tr`) and then 2,000 new ones (`x_new`, `y_new`). For degrees 1 to 12, fit on the training rows and store the training and held-out mean squared errors in the lists `train_mse` and `test_mse`. Plot both against degree and store the degree with the lowest held-out error as `best_degree`.

In [ ]:
rng = np.random.default_rng(17)
# YOUR CODE HERE
train_mse = test_mse = best_degree = None

In [ ]:
assert train_mse is not None and len(train_mse) == 12 and len(test_mse) == 12, "one error per degree, 1 to 12"
assert all(b <= a + 1e-9 for a, b in zip(train_mse, train_mse[1:])), \
    "training error should never rise with the degree: score each fit on the rows it was fitted to"
assert best_degree == int(np.argmin(test_mse)) + 1, "best_degree is the position of the lowest held-out error, plus 1"
assert 2 <= best_degree <= 9 and test_mse[-1] > min(test_mse), \
    "held-out error should fall and then turn back up: score on the 2,000 new hospitals, not the training ones"
print(f"best degree {best_degree}: held-out MSE {min(test_mse):.3f}; degree 12 {test_mse[-1]:.3f}")
print("Left of the turn: bias, too simple. Right of it: variance, fitting the noise.")

---
## 7. Choose by cross-validation, then report a fair error (PS4 Step 3.2)

Cross-validation scores each candidate on five folds instead of one split, and the spread across folds gives a rough interval: mean ± 2 × sd / √5. But the smallest of ten noisy CV scores is optimistic: it was picked partly because it got lucky. **Nested** cross-validation fixes that: an outer loop holds out a fold, an inner loop chooses the degree using only the outer training rows, and the chosen model is then scored on the held-out fold. The mean of the outer scores estimates the error of the whole procedure, choosing included (Lab 15 Part 4).

**Watch:** run this and read it.

In [ ]:
def fold_errors(x, y, d, folds):
    """The held-out MSE of a degree-d fit on each fold."""
    errs = []
    for a, b in folds.split(x):
        c = np.polyfit(x[a], y[a], d)
        errs.append(np.mean((np.polyval(c, x[b]) - y[b]) ** 2))
    return np.array(errs)

kf = KFold(n_splits=5, shuffle=True, random_state=0)
cv = [fold_errors(xf, yf, d, kf).mean() for d in range(1, 11)]
pick = int(np.argmin(cv)) + 1
e = fold_errors(xf, yf, pick, kf)
print(f"CV picks degree {pick}: MSE {e.mean():.3f}, rough interval "
      f"{e.mean() - 2 * e.std(ddof=1) / np.sqrt(5):.3f} to {e.mean() + 2 * e.std(ddof=1) / np.sqrt(5):.3f}")

outer = KFold(n_splits=5, shuffle=True, random_state=0)
scores = []
for a, b in outer.split(xf):
    inner = KFold(n_splits=5, shuffle=True, random_state=1)
    d = int(np.argmin([fold_errors(xf[a], yf[a], k, inner).mean() for k in range(1, 11)])) + 1   # chosen on a only
    c = np.polyfit(xf[a], yf[a], d)
    scores.append(np.mean((np.polyval(c, xf[b]) - yf[b]) ** 2))                                   # scored on b
print(f"nested estimate: {np.mean(scores):.3f}, against the best CV score {min(cv):.3f}")

**Your turn:** on your 80 training hospitals (`x_tr`, `y_tr`), use `KFold(n_splits=5, shuffle=True, random_state=0)` to store the mean CV error of degrees 1 to 10 in the list `cv_mse`, the best degree as `best_cv_degree`, and its rough interval as the tuple `cv_interval = (low, high)`. Then run nested CV: outer folds with `random_state=0`, inner folds with `random_state=1`, degrees 1 to 10. Store the five outer scores in `outer_scores`, the degree each outer fold chose in `chosen`, and their mean as `nested_mse`. You may write your own version of `fold_errors`.

In [ ]:
# YOUR CODE HERE
cv_mse = best_cv_degree = cv_interval = None
outer_scores = chosen = nested_mse = None

In [ ]:
assert cv_mse is not None and len(cv_mse) == 10, "cv_mse needs one mean CV error per degree, 1 to 10"
assert best_cv_degree == int(np.argmin(cv_mse)) + 1, "best_cv_degree is the position of the lowest CV error, plus 1"
assert cv_interval is not None and cv_interval[0] < cv_mse[best_cv_degree - 1] < cv_interval[1], \
    "cv_interval should be (low, high) around the best degree's CV error"
assert outer_scores is not None and len(outer_scores) == 5 and np.isclose(nested_mse, np.mean(outer_scores)), \
    "nested_mse is the mean of the five outer-fold scores"
assert chosen is not None and all(1 <= d <= 10 for d in chosen), "chosen holds the degree each outer fold picked"
assert nested_mse > min(cv_mse), \
    "with these seeds the nested estimate should come out above the best CV score: choose inside the outer training rows only"
print(f"CV picks degree {best_cv_degree}: {cv_mse[best_cv_degree - 1]:.3f} "
      f"(rough interval {cv_interval[0]:.3f} to {cv_interval[1]:.3f})")
print(f"nested estimate {nested_mse:.3f}; outer folds chose degrees {chosen}")
print("Report the nested number, and say it covers choosing the degree as well as fitting it.")

---
## 8. Fit the preprocessing inside each fold (PS4 Task 4.2)

Cross-validation is only fair if the held-out fold plays no part in building the model it scores. A step that learns from the data, such as `SelectKBest` choosing the columns most correlated with the outcome, has to be refitted on each fold's training rows. Run it once on all the rows and the held-out rows have already helped pick the columns. A `Pipeline` chains the steps into one model, so `cross_val_score` refits every step inside every fold. Fitted on all rows, the scaler alone does no harm for OLS with an intercept.

**Watch:** run this and read it.

In [ ]:
rng = np.random.default_rng(18)
n_plots = 100
plots = pd.DataFrame(np.column_stack([rng.uniform(5, 100, n_plots), rng.normal(600, 100, n_plots),
                                      rng.normal(size=(n_plots, 150))]),
                     columns=["area", "rain"] + [f"noise_{j}" for j in range(150)])   # 150 pure-noise columns
crop8 = 2 * plots["area"] + 0.3 * plots["rain"] + rng.normal(0, 60, n_plots)

# Selected on all rows: scale and pick 15 columns using ALL 100 farms, then cross-validate only the regression
picked = SelectKBest(f_regression, k=15).fit_transform(StandardScaler().fit_transform(plots), crop8)
all_rows_farm = cross_val_score(LinearRegression(), picked, crop8, cv=5, scoring="r2").mean()

# Inside the folds: the same three steps as one model, refitted on each fold's training rows
farm_pipe = Pipeline([("scale", StandardScaler()),
                      ("select", SelectKBest(f_regression, k=15)),
                      ("ols", LinearRegression())])
fold_farm = cross_val_score(farm_pipe, plots, crop8, cv=5, scoring="r2").mean()
print(f"selected on all rows: R-squared {all_rows_farm:.3f} | selected inside each fold: {fold_farm:.3f}")

**Your turn:** for 100 hospitals, build a DataFrame `wide` with `beds` (uniform 20 to 400), `staff_ratio` (Normal(1.5, 0.3)) and 250 columns of pure noise, and make `visits8 = 10 * beds + 2000 * staff_ratio + Normal(0, 1200)`. First the leaky way: scale and keep the 30 best columns using all rows, then `cross_val_score` a `LinearRegression` on them (cv=5, R-squared) and store the mean as `leaky_r2`. Then build `pipe`, a `Pipeline` of the same three steps, cross-validate it on `wide` the same way, and store the mean as `pipe_r2` and `leaky_r2 - pipe_r2` as `leak_gap`.

In [ ]:
rng = np.random.default_rng(19)
# YOUR CODE HERE
wide = visits8 = pipe = None
leaky_r2 = pipe_r2 = leak_gap = None

In [ ]:
assert wide is not None and wide.shape == (100, 252), "wide needs 100 rows: beds, staff_ratio and 250 noise columns"
assert isinstance(pipe, Pipeline), "pipe should be a sklearn Pipeline: Pipeline([(name, step), ...])"
assert any(isinstance(step, SelectKBest) for _, step in pipe.steps), "the column selection has to be one of the pipeline's steps"
assert leaky_r2 is not None and pipe_r2 is not None, "store both cross-validated R-squared means"
assert pipe_r2 < leaky_r2 - 0.1, \
    "the pipeline score should come out clearly lower: cross-validate the Pipeline on wide, not on the pre-selected columns"
assert np.isclose(leak_gap, leaky_r2 - pipe_r2), "leak_gap = leaky_r2 - pipe_r2"
print(f"selected on all rows: {leaky_r2:.3f} | inside each fold: {pipe_r2:.3f} | gap {leak_gap:.3f}")
print("Only the second number says what the whole procedure does on rows it has not seen.")
print("Below zero means the model predicts held-out hospitals worse than their plain average would.")

---
**Done.** R-squared in and out of sample, what extra columns do to it, residuals that fan out, an over-forecast on the largest units, a near-duplicate column, a complexity sweep, a cross-validated choice with a nested estimate, and preprocessing fitted inside each fold: the building blocks of PS4 Phases 1–3 and Task 4.2. Now open `ps4-starter.ipynb`.